In [1]:
import os, sys
sys.path.append(os.getenv("FALCON_PATH"))
print("sys.path:", sys.path)

sys.path: ['/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python314.zip', '/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14', '/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload', '', '/Users/ajayumashankar25gmail.com/Desktop/CAPSTONE/implementation/venv/lib/python3.14/site-packages', '/opt/homebrew/opt/python-tk@3.14/libexec', '/Users/ajayumashankar25gmail.com/Desktop/CAPSTONE/implementation/falcon_py']


In [2]:
import json
from falcon import Falcon

model = Falcon(512)

In [3]:
def serialize_command(command):
    return json.dumps(
        command, separators=(',', ':'),
        sort_keys=True
    ).encode('utf-8')

In [4]:
class Drone:
    def __init__(self, drone_id):
        self.drone_id = drone_id
        self.secret_key, self.public_key = model.keygen()

    def sign_command(self, command):
        searialized_command = serialize_command(command)
        signature = model.sign(self.secret_key, searialized_command)
        return signature

In [5]:
class TrustRegistry:
    def __init__(self):
        self.registry = {}

    def register_drone(self, drone:Drone):
        # store the Drone object so we can access its public key when verifying
        self.registry[drone.drone_id] = drone
    
    def verify(self, drone_id:str, command:dict, signature) -> bool:
        if drone_id not in self.registry:
            raise Exception("Unidentified Drone:", drone_id)
        
        drone = self.registry[drone_id]
        public_key = drone.public_key
        serialized = serialize_command(command)
        return model.verify(public_key, serialized, signature)

In [6]:
drone1 = Drone("drone1")
drone2 = Drone("drone2")
drone3 = Drone("drone3")
attacker_drone = Drone("attacker")

In [7]:
registry = TrustRegistry()
registry.register_drone(drone1)
registry.register_drone(drone2)
registry.register_drone(drone3)

print("Trusted drones:", list(registry.registry.keys()))

Trusted drones: ['drone1', 'drone2', 'drone3']


In [8]:
command = {"action": "takeoff",
           "altitude": 100,
           "speed": 10}

for i, drone in registry.registry.items():
    try:
        signature = drone.sign_command(command)
        registry.verify(i, command, signature)
        print("VALID TRUSTED SIGNATURE → ACCEPTED for drone:", i)
    except Exception as e:
        print("INVALID SIGNATURE → REJECTED for drone:", i, "Error:", e)

VALID TRUSTED SIGNATURE → ACCEPTED for drone: drone1
VALID TRUSTED SIGNATURE → ACCEPTED for drone: drone2
VALID TRUSTED SIGNATURE → ACCEPTED for drone: drone3


In [9]:
attack_signature = attacker_drone.sign_command(command)

try:
    registry.verify(attacker_drone.drone_id, command, attack_signature)
    print("UNEXPECTED ACCTEPTANCE for attacker drone:", attacker_drone.drone_id)
except Exception as e:
    print("INVALID SIGNATURE → REJECTED for attacker drone:", attacker_drone.drone_id, "Error:", e)

INVALID SIGNATURE → REJECTED for attacker drone: attacker Error: ('Unidentified Drone:', 'attacker')
